# 🏗️ IaC Benchmark Dataset Builder
> Builds a curated Terraform dataset for LLM IaC-generation benchmarks  
> modelled after **IaC-Eval (NeurIPS 2024)**, **DPIaC-Eval**, and **Multi-IaC-Eval (arXiv 2509.05303)**.

## Pipeline overview
```
1. Clone / fetch repositories
2. Ethical licence filter
3. Collect .tf files → raw dataset
4. Size filter (LOC / token budget)
5. Syntax validation  (TFLint + HCL2 parse)
6. Deployability check (terraform init + terraform plan)
7. AWS targeting filter
8. Aggregate & export
```

### Citation
> *"We use TFLint and Checkov to ensure that all source IaC templates used as the basis of IaC mutation meet standards of IaC best practices and security."*  
> — Davidson et al., Multi-IaC-Eval (2025), §3.5


In [79]:
# ── 1. Install dependencies ────────────────────────────────────────────────────
# Run once; comment out after first execution.
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

pip('requests', 'python-hcl2', 'tiktoken', 'pandas', 'tqdm',
    'PyYAML', 'gitpython', 'boto3', 'tabulate', 'rich')

print('✅ Python dependencies installed.')

# ── External tools (install once on system) ────────────────────────────────────
# terraform : https://developer.hashicorp.com/terraform/downloads
# tflint    : https://github.com/terraform-linters/tflint
# checkov   : pip install checkov
# All three must be on $PATH for cells 5 and 6.
for tool in ['terraform', 'tflint', 'checkov']:
    rc = subprocess.run(['which', tool], capture_output=True).returncode
    status = '✅' if rc == 0 else '⚠️  NOT FOUND – install before running cells 5/6'
    print(f'{tool:12s} {status}')


✅ Python dependencies installed.
terraform    ✅
tflint       ✅
checkov      ✅


In [ ]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
import os, pathlib

%env AWS_PROFILE=localstack
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = pathlib.Path('./iac_benchmark')          # root workspace
REPOS_DIR  = BASE_DIR / 'repos'                       # git clones land here
DATASET_DIR = BASE_DIR / 'dataset'                    # output CSVs
TMP_DIR    = BASE_DIR / 'tmp_plan'                    # isolated terraform plan dirs
for d in [REPOS_DIR, DATASET_DIR, TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── GitHub API token (optional – raises rate limit from 60 to 5000 req/hr) ───
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')     # set in env or paste here
GH_HEADERS   = {'Authorization': f'token {GITHUB_TOKEN}'} if GITHUB_TOKEN else {}

# ── AWS credentials (required for cell 6 – terraform plan) ───────────────────
# Best practice: export AWS_PROFILE=my-sandbox before launching Jupyter.
# A read-only IAM user with no deployment rights is sufficient for `terraform plan`.
AWS_REGION = os.environ.get('AWS_DEFAULT_REGION', 'ap-southeast-2')  # Sydney

# ── Size-filter thresholds (cell 4) ──────────────────────────────────────────
MIN_LOC    = 5        # discard near-empty stubs
MAX_LOC    = 600      # upper bound used by DPIaC-Eval (Level 5 ≈ 200 LoC, we add buffer)
MAX_TOKENS = 8_000    # ~8 k-token context budget leaves room for prompt + response

# ── Token encoder (for LLM token counting) ───────────────────────────────────
import tiktoken
TOKENIZER = tiktoken.get_encoding('cl100k_base')   # GPT-4 / Claude compatible

print('Configuration loaded.')
print(f'  BASE_DIR  : {BASE_DIR.resolve()}')
print(f'  AWS_REGION: {AWS_REGION}')
print(f'  LOC range : [{MIN_LOC}, {MAX_LOC}]  |  max tokens: {MAX_TOKENS}')


env: AWS_PROFILE=localstack
Configuration loaded.
  BASE_DIR  : /Users/iksena/Documents/research/data_analysis/iac_benchmark
  AWS_REGION: ap-southeast-2
  LOC range : [5, 600]  |  max tokens: 8000


In [82]:
# ── 3. Repository Registry ────────────────────────────────────────────────────
# Each entry carries:
#   owner/repo, clone_url, subdirectory to scan ('' = whole repo),
#   primary cloud target, and the licence string reported on GitHub.
#
# Licence eligibility follows Multi-IaC-Eval & IaC-Eval practice:
# only OSI-approved permissive licences that allow redistribution
# in a research dataset are accepted (cell 3b performs the API check).

REPO_REGISTRY = [
    dict(
        slug='hashicorp/terraform-guides',
        subdir='',
        cloud='multi',
        category='official',
        description='HashiCorp official usage patterns across providers'
    ),
    dict(
        slug='GoogleCloudPlatform/terraform-google-examples',
        subdir='',
        cloud='gcp',
        category='official',
        description='GCP examples: GKE, SQL, Vault'
    ),
    dict(
        slug='aws-samples/aws-terraform-best-practices',
        subdir='',
        cloud='aws',
        category='official',
        description='AWS architecture best practices – modularised .tf'
    ),
    dict(
        slug='terraform-aws-modules/terraform-aws-vpc',
        subdir='examples',
        cloud='aws',
        category='official',
        description='Complete, deployable VPC configurations'
    ),
    dict(
        slug='futurice/terraform-examples',
        subdir='',
        cloud='multi',
        category='educational',
        description='Copy-paste recipes for AWS and Azure'
    ),
    dict(
        slug='brokedba/terraform-examples',
        subdir='',
        cloud='multi',
        category='educational',
        description='OCI, AWS, Azure, GCP, Alicloud – broad multi-cloud'
    ),
    dict(
        slug='ContainerSolutions/terraform-examples',
        subdir='',
        cloud='multi',
        category='educational',
        description='Simple, idiomatic – single-file per example (ideal Pass@1)'
    ),
    dict(
        slug='ned1313/terraform-tuesdays',
        subdir='',
        cloud='multi',
        category='educational',
        description='Weekly demos for AWS and Azure'
    ),
    dict(
        slug='alfonsof/terraform-aws-examples',
        subdir='',
        cloud='aws',
        category='specialised',
        description='Step-by-step from single server to full cluster + LB + ASG'
    ),
    dict(
        slug='alfonsof/terraform-azure-examples',
        subdir='',
        cloud='azure',
        category='specialised',
        description='Azure VMs, NSGs, resource management'
    ),
    dict(
        slug='databricks/terraform-databricks-examples',
        subdir='',
        cloud='multi',
        category='specialised',
        description='Databricks data-platform across AWS, Azure, GCP'
    ),
    dict(
        slug='aws-cloudformation/iac-model-evaluation',
        subdir='terraform',
        cloud='aws',
        category='benchmark-source',
        description='High-quality IaC samples specifically curated for LLM evaluation, excluding training data'
    ),
    dict(
        slug='autoiac-project/iac-eval',
        subdir='dataset/terraform',
        cloud='aws',
        category='benchmark-source',
        description='A human-curated dataset of 458 Terraform questions ranging from simple to complex'
    ),
    dict(
        slug='gitmurali/terraform-aws-snippets',
        subdir='',
        cloud='aws',
        category='educational',
        description='Clean, single-directory snippets for VPC, RDS with Secrets Manager, and ECS Fargate'
    ),
    dict(
        slug='garutilorenzo/aws-terraform-examples',
        subdir='examples',
        cloud='aws',
        category='educational',
        description='Provisioning AWS resources using simple main.tf entrypoints for modules'
    ),
    dict(
        slug='ValeriiVasianovych/terraform-iac',
        subdir='',
        cloud='aws',
        category='educational',
        description='Learning resource with practical single-file examples for AWS infrastructure tasks'
    ),
    dict(
        slug='antonputra/tutorials',
        subdir='lessons/040',
        cloud='aws',
        category='educational',
        description='Beginner-friendly single-file examples for EC2, IAM, and Remote State configuration'
    ),
    dict(
        slug='ovotech/terraform-testing-samples',
        subdir='',
        cloud='aws',
        category='best-practices',
        description='Production-grade code samples focusing on IaC testing and verification steps'
    ),
    dict(
        slug='zealvora/terraform-beginner-to-advanced-resource',
        subdir='',
        cloud='aws',
        category='educational',
        description='Course resources with atomic .tf files for EC2, VPC, and Security Groups'
    ),
    dict(
        slug='wardviaene/terraform-course',
        subdir='',
        cloud='aws',
        category='educational',
        description='Demo-based single-file configurations for ELB, AutoScaling, and VPC'
    ),
    dict(
        slug='diodonfrost/terraform-aws-examples',
        subdir='',
        cloud='aws',
        category='educational',
        description='A collection of independent AWS manifests for fast deployment testing'
    ),
    dict(
        slug='brikis98/terraform-up-and-running-code',
        subdir='code/terraform/02-intro-to-terraform-syntax',
        cloud='aws',
        category='educational',
        description='Reference code for single-file Terraform syntax and basic AWS resources'
    ),
    dict(
        slug='aws-samples/cict-terraform-scripts',
        subdir='terraform',
        cloud='aws',
        category='best-practices',
        description='AWS-curated scripts demonstrating verified root-module patterns'
    ),
    dict(
        slug='stack72/terraform-examples',
        subdir='',
        cloud='multi',
        category='educational',
        description='Simple, single-template examples covering AWS, Azure, and core providers'
    )
]

print(f'Registered {len(REPO_REGISTRY)} repositories.')


Registered 24 repositories.


In [84]:
# ── 3b. Ethical Licence Filter ────────────────────────────────────────────────
# Query the GitHub Licence API for each repository.
# Only OSI-approved permissive licences are kept in the pipeline.
# Copyleft (GPL) and proprietary licences are flagged and excluded.

import requests, json
from rich.console import Console
from rich.table import Table

# OSI-approved licences permissible in a research / redistribution context
ALLOWED_SPDX = {
    'MIT', 'MIT-0', 'Apache-2.0', 'BSD-2-Clause', 'BSD-3-Clause',
    'ISC', 'MPL-2.0', 'CC0-1.0', 'Unlicense', 'WTFPL'
}
# Copyleft / viral licences – included in benchmark ONLY if licence explicitly
# permits use in academic datasets (check repo NOTICE/README manually)
COPYLEFT_SPDX = {'GPL-2.0', 'GPL-3.0', 'LGPL-2.1', 'LGPL-3.0', 'AGPL-3.0'}

def fetch_licence(slug: str) -> dict:
    url = f'https://api.github.com/repos/{slug}/license'
    r = requests.get(url, headers=GH_HEADERS, timeout=15)
    if r.status_code == 200:
        data = r.json()
        spdx = data.get('license', {}).get('spdx_id', 'NOASSERTION')
        name = data.get('license', {}).get('name', 'Unknown')
        return {'spdx': spdx, 'name': name, 'html_url': data.get('html_url', '')}
    elif r.status_code == 404:
        return {'spdx': 'NONE', 'name': 'No licence file found', 'html_url': ''}
    else:
        return {'spdx': 'ERROR', 'name': f'HTTP {r.status_code}', 'html_url': ''}

console = Console()
table = Table(title='Licence Eligibility Check', show_lines=True)
table.add_column('Repository', style='cyan')
table.add_column('SPDX', style='yellow')
table.add_column('Eligible?', style='bold')
table.add_column('Notes')

APPROVED_REPOS = []
EXCLUDED_REPOS = []

for repo in REPO_REGISTRY:
    slug = repo['slug']
    lic  = fetch_licence(slug)
    repo['licence_spdx'] = lic['spdx']
    repo['licence_name'] = lic['name']

    if lic['spdx'] in ALLOWED_SPDX:
        eligible  = '✅ YES'
        note      = 'Permissive – clear for redistribution'
        repo['licence_eligible'] = True
        APPROVED_REPOS.append(repo)
    elif lic['spdx'] in COPYLEFT_SPDX:
        eligible  = '⚠️  COPYLEFT'
        note      = 'Review NOTICE/README for dataset clause'
        repo['licence_eligible'] = True
        APPROVED_REPOS.append(repo)
    else:
        eligible  = '❌ EXCLUDED'
        note      = f'No/unknown licence ({lic["spdx"]})'
        repo['licence_eligible'] = False
        EXCLUDED_REPOS.append(repo)

    table.add_row(slug, lic['spdx'], eligible, note)

console.print(table)
print(f'\nApproved: {len(APPROVED_REPOS)} | Excluded: {len(EXCLUDED_REPOS)}')
print('\nManually override repo["licence_eligible"] = True for copyleft repos')
print('after confirming that the licence allows academic dataset redistribution.')


                                             Licence Eligibility Check                                             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Repository                                ┃ SPDX        ┃ Eligible?   ┃ Notes                                   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ hashicorp/terraform-guides                │ MPL-2.0     │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ GoogleCloudPlatform/terraform-google-exa… │ Apache-2.0  │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ aws-samples/aws-terraform-best-practices  │ NOASSERTION │ ❌ EXCLUDED │ No/unknown licence (NOASSERTION)        │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ terraform-aws-modules/terraform-aws-vpc   │ Apache-2.0  │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ futurice/terraform-examples               │ MIT         │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ brokedba/terraform-examples               │ GPL-3.0     │ ⚠️  COPYLEFT │ Review NOTICE/README for dataset clause │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ ContainerSolutions/terraform-examples     │ NONE        │ ❌ EXCLUDED │ No/unknown licence (NONE)               │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ ned1313/terraform-tuesdays                │ MIT         │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ alfonsof/terraform-aws-examples           │ MIT         │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ alfonsof/terraform-azure-examples         │ MIT         │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ databricks/terraform-databricks-examples  │ NOASSERTION │ ❌ EXCLUDED │ No/unknown licence (NOASSERTION)        │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ aws-cloudformation/iac-model-evaluation   │ MIT-0       │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ autoiac-project/iac-eval                  │ MIT         │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ gitmurali/terraform-aws-snippets          │ MIT         │ ✅ YES      │ Permissive – clear for redistribution   │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ garutilorenzo/aws-terraform-examples      │ GPL-3.0     │ ⚠️  COPYLEFT │ Review NOTICE/README for dataset clause │
├───────────────────────────────────────────┼─────────────┼─────────────┼─────────────────────────────────────────┤
│ ValeriiVasianovych/terraform-iac          │ MIT         │ ✅ YES  


Approved: 18 | Excluded: 6

Manually override repo["licence_eligible"] = True for copyleft repos
after confirming that the licence allows academic dataset redistribution.


In [86]:
# ── 4. Clone / Update Repositories ───────────────────────────────────────────
# Uses shallow clone (depth=1) to save disk space.
# If the directory already exists, pulls the latest changes instead.

import git
from tqdm.notebook import tqdm

def clone_or_pull(slug: str, dest: pathlib.Path) -> bool:
    url = f'https://github.com/{slug}.git'
    try:
        if dest.exists():
            r = git.Repo(dest)
            r.remotes.origin.pull(depth=1)
            return True
        else:
            git.Repo.clone_from(url, dest, depth=1, single_branch=True)
            return True
    except Exception as e:
        print(f'  ⚠️  {slug}: {e}')
        return False

eligible = [r for r in REPO_REGISTRY if r.get('licence_eligible')]
print(f'Cloning {len(eligible)} approved repositories…\n')

for repo in tqdm(eligible, desc='Cloning'):
    slug   = repo['slug']
    dest   = REPOS_DIR / slug.replace('/', '__')
    repo['local_path'] = str(dest)
    ok = clone_or_pull(slug, dest)
    print(f'  {"✅" if ok else "❌"} {slug}')

print('\nClone phase complete.')


Cloning 18 approved repositories…



Cloning:   0%|          | 0/18 [00:00<?, ?it/s]

  ✅ hashicorp/terraform-guides
  ✅ GoogleCloudPlatform/terraform-google-examples
  ✅ terraform-aws-modules/terraform-aws-vpc
  ✅ futurice/terraform-examples
  ✅ brokedba/terraform-examples
  ✅ ned1313/terraform-tuesdays
  ✅ alfonsof/terraform-aws-examples
  ✅ alfonsof/terraform-azure-examples
  ✅ aws-cloudformation/iac-model-evaluation
  ✅ autoiac-project/iac-eval
  ✅ gitmurali/terraform-aws-snippets
  ✅ garutilorenzo/aws-terraform-examples
  ✅ ValeriiVasianovych/terraform-iac
  ✅ antonputra/tutorials
  ✅ ovotech/terraform-testing-samples
  ✅ diodonfrost/terraform-aws-examples
  ✅ brikis98/terraform-up-and-running-code
  ✅ aws-samples/cict-terraform-scripts

Clone phase complete.


In [ ]:
# ── 4b. ALTERNATIVE – Fetch .tf files via GitHub API (no disk clone) ─────────
# Useful when you only want to sample files without cloning multi-GB repos.
# Skip this cell if you used cell 4 (clone).

import requests, base64, time

def api_get(url: str) -> dict | list | None:
    r = requests.get(url, headers=GH_HEADERS, timeout=20)
    if r.status_code == 403:                     # rate limited
        reset = int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
        wait  = max(reset - int(time.time()), 5)
        print(f'Rate-limited – waiting {wait}s…')
        time.sleep(wait)
        return api_get(url)
    return r.json() if r.ok else None

def list_tf_files(slug: str, path: str = '', ref: str = 'HEAD') -> list[dict]:
    """Recursively list all .tf files in a GitHub repo subtree."""
    url   = f'https://api.github.com/repos/{slug}/git/trees/{ref}?recursive=1'
    tree  = api_get(url)
    if not tree:
        return []
    files = [
        item for item in tree.get('tree', [])
        if item['type'] == 'blob'
        and item['path'].endswith('.tf')
        and (path == '' or item['path'].startswith(path))
    ]
    return files

def fetch_file_content(slug: str, sha: str) -> str:
    """Fetch raw file content by blob SHA."""
    url  = f'https://api.github.com/repos/{slug}/git/blobs/{sha}'
    blob = api_get(url)
    if not blob:
        return ''
    return base64.b64decode(blob['content']).decode('utf-8', errors='replace')

# Example: fetch files from one repo without cloning
# slug = 'ContainerSolutions/terraform-examples'
# files = list_tf_files(slug)
# content = fetch_file_content(slug, files[0]['sha'])
print('API-fetch helpers defined. Use list_tf_files() / fetch_file_content() as needed.')


In [88]:
# ── 5. Collect .tf Files → Raw Dataset ───────────────────────────────────────
# Walks approved cloned repos and harvests every .tf file.
# Records: path, source slug, cloud target, content, LOC, tokens.

import pandas as pd

RAW_RECORDS = []

def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text))

for repo in eligible:
    if 'local_path' not in repo:
        continue
    root = pathlib.Path(repo['local_path'])
    subdir = repo.get('subdir', '')
    scan_root = root / subdir if subdir else root

    for tf_path in scan_root.rglob('*.tf'):
        try:
            content = tf_path.read_text(encoding='utf-8', errors='replace')
        except Exception:
            continue

        loc    = content.count('\n') + 1
        tokens = count_tokens(content)
        # Relative path within repo (for reproducibility)
        rel    = str(tf_path.relative_to(root))

        RAW_RECORDS.append({
            'source_slug'       : repo['slug'],
            'source_category'   : repo['category'],
            'primary_cloud'     : repo['cloud'],
            'licence_spdx'      : repo['licence_spdx'],
            'file_path'         : rel,
            'github_url'        : f'https://github.com/{repo["slug"]}/blob/HEAD/{rel}',
            'content'           : content,
            'loc'               : loc,
            'tokens'            : tokens,
        })

df_raw = pd.DataFrame(RAW_RECORDS)
print(f'Raw collection: {len(df_raw):,} .tf files')
print(df_raw[['source_slug','primary_cloud','loc','tokens']]
      .groupby('source_slug').agg({'loc':'mean', 'tokens':'mean', 'primary_cloud':'first'})
      .round(1).to_string())


Raw collection: 1,713 .tf files
                                                 loc  tokens primary_cloud
source_slug                                                               
GoogleCloudPlatform/terraform-google-examples  103.4   610.9           gcp
ValeriiVasianovych/terraform-iac                28.8   201.6           aws
alfonsof/terraform-aws-examples                 19.2   125.2           aws
alfonsof/terraform-azure-examples               52.6   376.1         azure
antonputra/tutorials                            13.2    73.2           aws
aws-cloudformation/iac-model-evaluation         35.2   256.7           aws
aws-samples/cict-terraform-scripts              35.9   227.5           aws
brikis98/terraform-up-and-running-code          46.6   263.9           aws
brokedba/terraform-examples                     70.0   597.4         multi
diodonfrost/terraform-aws-examples              31.0   209.0           aws
futurice/terraform-examples                     51.0   394.5        

In [90]:
# ── 5b. Single-Root Template Normalisation ────────────────────────────────────
# A "single-root" Terraform template is a directory that contains exactly one
# logical configuration root — i.e. all .tf files in the same folder form
# one deployable unit (one terraform init / plan context).
#
# This cell:
#   1. Groups collected files by their parent directory (= one Terraform root)
#   2. Keeps only single-file roots  OR  merges multi-file roots into one
#      concatenated "main.tf" string so every dataset record is self-contained
#   3. Copies qualifying templates into  BASE_DIR/single_root_templates/<slug>/
#   4. Adds a `root_path` and `is_merged` column to df_raw

import shutil
import re
from pathlib import Path

SINGLE_ROOT_DIR = BASE_DIR / 'single_root_templates'
SINGLE_ROOT_DIR.mkdir(parents=True, exist_ok=True)

# ── Tunables ──────────────────────────────────────────────────────────────────
MAX_FILES_PER_ROOT   = 10    # roots with more .tf files are likely framework-level
                             # modules (too complex for a standalone benchmark item)
MERGE_MULTI_FILE     = True  # True  → concatenate all files in a root into one
                             # False → keep only directories with exactly 1 .tf file

# ── Helper: is this a "test / fixture" path we should skip? ──────────────────
_SKIP_DIRS = re.compile(
    r'(^|/)(\.|__pycache__|\.git|\.terraform|\.terragrunt-cache'
    r'|tests?|fixtures?|testdata|vendor|node_modules)(/|$)',
    re.IGNORECASE
)

def _should_skip(rel_path: str) -> bool:
    return bool(_SKIP_DIRS.search(rel_path))

# ── Group raw records by (source_slug, parent_dir) ───────────────────────────
df_raw['parent_dir'] = df_raw['file_path'].apply(
    lambda p: str(Path(p).parent)          # '' means repo root itself
)

groups = df_raw[~df_raw['file_path'].apply(_should_skip)].groupby(
    ['source_slug', 'parent_dir']
)

normalised_records = []
copy_stats = {'kept_single': 0, 'merged': 0, 'skipped_large': 0, 'skipped_test': 0}

for (slug, parent), group in tqdm(groups, desc='Normalising roots'):

    # ── Skip oversized roots (framework-level modules, not examples) ──────────
    if len(group) > MAX_FILES_PER_ROOT:
        copy_stats['skipped_large'] += 1
        continue

    # ── Merge or single-file decision ─────────────────────────────────────────
    if len(group) == 1 or not MERGE_MULTI_FILE:
        # Single file – use as-is; if MERGE_MULTI_FILE=False skip multi-file roots
        if len(group) > 1 and not MERGE_MULTI_FILE:
            copy_stats['skipped_test'] += 1
            continue
        row        = group.iloc[0]
        merged_src = row['content']
        is_merged  = False
        origin_files = [row['file_path']]
    else:
        # ── Merge: sort files so main.tf / variables.tf / outputs.tf come first ─
        priority = {'main.tf': 0, 'variables.tf': 1, 'outputs.tf': 2,
                    'providers.tf': 3, 'versions.tf': 4, 'locals.tf': 5}
        ordered = sorted(
            group.itertuples(),
            key=lambda r: (priority.get(Path(r.file_path).name, 99), r.file_path)
        )
        sections = []
        for r in ordered:
            fname = Path(r.file_path).name
            sections.append(f'# ── {fname} ────────────────────────────────────\n{r.content}')
        merged_src   = '\n\n'.join(sections)
        is_merged    = True
        origin_files = [r.file_path for r in ordered]
        copy_stats['merged'] += 1

    # ── Compute aggregate metrics for the merged/single root ─────────────────
    row0        = group.iloc[0]
    loc         = merged_src.count('\n') + 1
    tokens      = count_tokens(merged_src)
    folder_name = (parent.replace('/', '__').replace(' ', '_') or 'ROOT')
    dest_slug   = slug.replace('/', '__')

    # ── Write to flat output folder ───────────────────────────────────────────
    dest_dir  = SINGLE_ROOT_DIR / dest_slug
    dest_dir.mkdir(parents=True, exist_ok=True)
    safe_name = f'{folder_name}.tf'
    dest_file = dest_dir / safe_name
    dest_file.write_text(merged_src, encoding='utf-8')

    copy_stats['kept_single'] += 1

    normalised_records.append({
        'source_slug'    : slug,
        'source_category': row0['source_category'],
        'primary_cloud'  : row0['primary_cloud'],
        'licence_spdx'   : row0['licence_spdx'],
        'root_path'      : str(Path(slug) / parent) if parent else slug,
        'origin_files'   : ' | '.join(origin_files),
        'file_count'     : len(origin_files),
        'is_merged'      : is_merged,
        'dest_file'      : str(dest_file.relative_to(BASE_DIR)),
        'github_url'     : f'https://github.com/{slug}/tree/HEAD/{parent}' if parent else f'https://github.com/{slug}',
        'content'        : merged_src,
        'loc'            : loc,
        'tokens'         : tokens,
    })

df_roots = pd.DataFrame(normalised_records)

# ── Write index CSV ───────────────────────────────────────────────────────────
index_path = SINGLE_ROOT_DIR / 'index.csv'
df_roots.drop(columns=['content']).to_csv(index_path, index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 60)
print('SINGLE-ROOT NORMALISATION SUMMARY')
print('=' * 60)
print(f"  Input .tf records         : {len(df_raw):>6,}")
print(f"  Unique roots found        : {len(groups):>6,}")
print(f"  ✅  Kept (single-file)    : {copy_stats['kept_single'] - copy_stats['merged']:>6,}")
print(f"  🔀  Merged (multi-file)   : {copy_stats['merged']:>6,}")
print(f"  ⏭️   Skipped (>={MAX_FILES_PER_ROOT} files) : {copy_stats['skipped_large']:>6,}")
print(f"  Output templates          : {len(df_roots):>6,}")
print(f"\nOutput folder : {SINGLE_ROOT_DIR.resolve()}")
print(f"Index CSV     : {index_path}")
print()
print('Top 5 sources by template count:')
print(df_roots.groupby('source_slug')['root_path']
      .count().sort_values(ascending=False).head(5).to_string())
print()
print('Merged vs single-file breakdown:')
print(df_roots['is_merged'].value_counts().rename({True: 'merged', False: 'single-file'}).to_string())

# ── Replace df_raw with the normalised set for all downstream cells ───────────
# Comment this out if you want to keep both the raw and normalised dataframes.
df_raw = df_roots.copy()
print('\n✅ df_raw replaced with normalised single-root dataset.')
print('   All downstream cells (size filter, syntax check, plan) will use it.')


Normalising roots:   0%|          | 0/509 [00:00<?, ?it/s]

SINGLE-ROOT NORMALISATION SUMMARY
  Input .tf records         :  1,713
  Unique roots found        :    509
  ✅  Kept (single-file)    :    157
  🔀  Merged (multi-file)   :    336
  ⏭️   Skipped (>=10 files) :     16
  Output templates          :    493

Output folder : /Users/iksena/Documents/research/data_analysis/iac_benchmark/single_root_templates
Index CSV     : iac_benchmark/single_root_templates/index.csv

Top 5 sources by template count:
source_slug
ned1313/terraform-tuesdays          228
ValeriiVasianovych/terraform-iac     68
hashicorp/terraform-guides           45
alfonsof/terraform-aws-examples      35
brokedba/terraform-examples          30

Merged vs single-file breakdown:
is_merged
merged         336
single-file    157

✅ df_raw replaced with normalised single-root dataset.
   All downstream cells (size filter, syntax check, plan) will use it.


In [92]:
# ── 6. Size Filter ────────────────────────────────────────────────────────────
# Mirrors the DPIaC-Eval difficulty stratification:
#   Level 1: <50 LoC / 1-2 resources
#   Level 2: 50-100 LoC
#   Level 3: 100-150 LoC
#   Level 4: 150-200 LoC
#   Level 5: >200 LoC / 12-14 resources
#
# We also enforce a MAX_TOKENS budget so templates fit within an LLM context.
# IaC-Eval (NeurIPS 2024) notes longest configs can reach 280 LoC / 24 resources.

def assign_difficulty(loc: int) -> int:
    if loc < 50:  return 1
    if loc < 100: return 2
    if loc < 150: return 3
    if loc < 200: return 4
    return 5

df_raw['difficulty'] = df_raw['loc'].apply(assign_difficulty)

mask_loc    = df_raw['loc'].between(MIN_LOC, MAX_LOC)
mask_tokens = df_raw['tokens'] <= MAX_TOKENS

df_sized = df_raw[mask_loc & mask_tokens].copy()

print(f'After size filter: {len(df_sized):,} files (from {len(df_raw):,})')
print(f'  Removed (too short <{MIN_LOC} LoC) : {(~mask_loc).sum():,}')
print(f'  Removed (too long >{MAX_LOC} LoC)  : {(df_raw["loc"] > MAX_LOC).sum():,}')
print(f'  Removed (>{MAX_TOKENS} tokens)     : {(~mask_tokens).sum():,}')
print()
print('Difficulty distribution:')
print(df_sized['difficulty'].value_counts().sort_index().to_string())


After size filter: 483 files (from 493)
  Removed (too short <5 LoC) : 10
  Removed (too long >600 LoC)  : 9
  Removed (>8000 tokens)     : 1

Difficulty distribution:
difficulty
1    145
2    139
3     74
4     43
5     82


In [94]:
# ── 7. Syntax Validation ──────────────────────────────────────────────────────
# Two-stage:
#   a) python-hcl2 parse  – fast in-process HCL2 check (no external tool needed)
#   b) tflint             – provider-aware lint (requires tflint on $PATH)
#
# Multi-IaC-Eval uses TFLint + Checkov; IaC-Eval uses `terraform plan`.
# We mirror the Multi-IaC-Eval approach for the static-analysis layer.

import hcl2, io, subprocess, tempfile, json
from tqdm.notebook import tqdm
tqdm.pandas()

# ── a) python-hcl2 parse ──────────────────────────────────────────────────────
def hcl2_valid(content: str) -> tuple[bool, str]:
    try:
        hcl2.load(io.StringIO(content))
        return True, ''
    except Exception as e:
        return False, str(e)[:200]

# ── b) tflint (subprocess) ────────────────────────────────────────────────────
def tflint_check(content: str) -> tuple[bool, str]:
    """Write to a temp dir and run tflint --format=json."""
    with tempfile.TemporaryDirectory() as tmpd:
        tf_file = pathlib.Path(tmpd) / 'main.tf'
        tf_file.write_text(content, encoding='utf-8')
        result = subprocess.run(
            ['tflint', '--format=json', '--no-color'],
            cwd=tmpd, capture_output=True, text=True, timeout=30
        )
        try:
            out = json.loads(result.stdout or '{}')
            issues = out.get('issues', [])
            errors = [i for i in issues if i.get('rule', {}).get('severity') == 'error']
            if errors:
                msgs = '; '.join(e.get('message','') for e in errors[:3])
                return False, msgs
            return True, ''
        except json.JSONDecodeError:
            # tflint not found or returned non-JSON; fall back to return code
            return result.returncode == 0, result.stderr[:200]

TFLINT_AVAILABLE = subprocess.run(
    ['which', 'tflint'], capture_output=True).returncode == 0

print('Running HCL2 parse check…')
df_sized[['hcl2_valid', 'hcl2_error']] = (
    df_sized['content']
    .progress_apply(lambda c: pd.Series(hcl2_valid(c)))
)

if TFLINT_AVAILABLE:
    print('Running TFLint check…')
    df_sized[['tflint_pass', 'tflint_error']] = (
        df_sized['content']
        .progress_apply(lambda c: pd.Series(tflint_check(c)))
    )
else:
    print('⚠️  tflint not found – skipping TFLint column (hcl2_valid used only)')
    df_sized['tflint_pass']  = None
    df_sized['tflint_error'] = 'tflint not available'

n_hcl2_pass   = df_sized['hcl2_valid'].sum()
print(f'\nHCL2 parse  passed: {n_hcl2_pass:,} / {len(df_sized):,}')
if TFLINT_AVAILABLE:
    n_tflint_pass = df_sized['tflint_pass'].sum()
    print(f'TFLint      passed: {n_tflint_pass:,} / {len(df_sized):,}')


Running HCL2 parse check…


  0%|          | 0/483 [00:00<?, ?it/s]

Running TFLint check…


  0%|          | 0/483 [00:00<?, ?it/s]


HCL2 parse  passed: 480 / 483
TFLint      passed: 483 / 483


In [96]:
# ── 7b. YAML Validation for embedded YAML here-docs ──────────────────────────
# Some Terraform files embed YAML policies (e.g. aws_iam_policy inline_document,
# kubernetes_manifest). Extract and validate any <<-EOT ... EOT here-doc blocks.

import re, yaml

HEREDOC_RE = re.compile(
    r'<<[-~]?(?P<tag>[A-Z_]+)\s*\n(?P<body>.*?)\n(?P=tag)',
    re.DOTALL
)

def validate_embedded_yaml(content: str) -> tuple[bool, str]:
    """Return (all_valid, first_error). Passes if no YAML here-docs present."""
    for m in HEREDOC_RE.finditer(content):
        body = m.group('body')
        # Only attempt to parse if it looks like YAML (has : or - patterns)
        if ':' not in body and not body.strip().startswith('-'):
            continue
        try:
            yaml.safe_load(body)
        except yaml.YAMLError as e:
            return False, str(e)[:200]
    return True, ''

df_sized[['yaml_valid', 'yaml_error']] = (
    df_sized['content']
    .progress_apply(lambda c: pd.Series(validate_embedded_yaml(c)))
)

print(f'YAML embedded check passed: {df_sized["yaml_valid"].sum():,} / {len(df_sized):,}')

# ── Syntax-clean filter ───────────────────────────────────────────────────────
tflint_col   = 'tflint_pass' if TFLINT_AVAILABLE else 'hcl2_valid'
df_syntaxok  = df_sized[
    df_sized['hcl2_valid'] & df_sized[tflint_col].fillna(True) & df_sized['yaml_valid']
].copy()

print(f'\nSyntax-clean dataset: {len(df_syntaxok):,} files')


  0%|          | 0/483 [00:00<?, ?it/s]

YAML embedded check passed: 476 / 483

Syntax-clean dataset: 473 files


In [98]:
# ── 8. AWS Targeting Filter ───────────────────────────────────────────────────
# For the deployability check in cell 9 we need files that target AWS.
# Detection strategy:
#   1. source_cloud == 'aws'
#   2. OR content contains `provider "aws"` or `aws_` resource references
#
# Multi-cloud files (both aws_ and google_ resources) are tagged 'multi'
# but still included if they have at least one aws_ resource.

def detect_aws(content: str, declared_cloud: str) -> bool:
    if declared_cloud == 'aws':
        return True
    patterns = [
        r'provider\s+["\']aws["\']',
        r'resource\s+["\']aws_',
        r'data\s+["\']aws_',
        r'module\s+.*terraform-aws-modules',
    ]
    for pat in patterns:
        if re.search(pat, content):
            return True
    return False

df_syntaxok['targets_aws'] = df_syntaxok.apply(
    lambda r: detect_aws(r['content'], r['primary_cloud']), axis=1
)

df_aws = df_syntaxok[df_syntaxok['targets_aws']].copy()

print(f'AWS-targeting files: {len(df_aws):,} / {len(df_syntaxok):,} syntax-clean files')
print('\nCloud breakdown (syntax-clean):')
print(df_syntaxok['primary_cloud'].value_counts().to_string())


AWS-targeting files: 201 / 473 syntax-clean files

Cloud breakdown (syntax-clean):
primary_cloud
multi    310
aws      150
azure      7
gcp        6


In [103]:
# ── 9. Deployability Check (terraform plan) ───────────────────────────────────
# Mirrors the IaC-Eval evaluation workflow:
#   "The generated IaC program is fed into a two-phase pipeline …
#    using the native terraform plan command which produces a dependency graph."
#   — Kon et al., IaC-Eval (NeurIPS 2024), §2.1
#
# We run:
#   terraform init  -backend=false
#   terraform validate
#   terraform plan  -refresh=false
#
# ⚠️  terraform plan with the AWS provider sends read-only describe/list API
#     calls.  Use a restricted IAM role.  No resources are created or modified.
#
# ⚠️  This can be slow (~30-60 s per file due to provider initialisation).
#     Use DEPLOY_SAMPLE_N to limit the number of files checked, or set
#     SKIP_DEPLOY = True to skip this cell entirely.

import shutil, concurrent.futures

SKIP_DEPLOY   = False       # set True to skip (useful in offline/no-AWS mode)
DEPLOY_SAMPLE_N = 50        # check at most N files (stratified by difficulty)

def terraform_plan(content: str, slug: str, file_path: str) -> dict:
    with tempfile.TemporaryDirectory() as tmpd:
        tf = pathlib.Path(tmpd) / 'main.tf'
        # Inject a minimal provider block if none exists
        if 'provider "aws"' not in content:
            provider_block = (
                f'provider "aws" {{ region = "{AWS_REGION}" }}\n\n'
            )
            content = provider_block + content
        tf.write_text(content, encoding='utf-8')

        def run(cmd, **kwargs):
            return subprocess.run(
                cmd, cwd=tmpd, capture_output=True,
                text=True, timeout=120, **kwargs
            )

        # Step 1: init (downloads provider plugin on first call – cached thereafter)
        r_init = run(['terraform', 'init', '-backend=false',
                      '-input=false', '-no-color'])
        if r_init.returncode != 0:
            return {'init_ok': False, 'validate_ok': None, 'plan_ok': None,
                    'deploy_error': r_init.stderr[:400]}

        # Step 2: validate (syntax + provider schema)
        r_val = run(['terraform', 'validate', '-no-color', '-json'])
        try:
            val_json = json.loads(r_val.stdout or '{}')
            validate_ok = val_json.get('valid', r_val.returncode == 0)
            val_diags   = str([d.get('summary','') for d in val_json.get('diagnostics',[]) if d.get('severity')=='error'])[:300]
        except Exception:
            validate_ok = r_val.returncode == 0
            val_diags   = r_val.stderr[:300]

        if not validate_ok:
            return {'init_ok': True, 'validate_ok': False, 'plan_ok': None,
                    'deploy_error': val_diags}

        # Step 3: plan (dry-run – no actual resource creation)
        r_plan = run([
            'terraform', 'plan', '-refresh=false',
            '-input=false', '-no-color', '-json', '-out=/dev/null'
        ])
        plan_ok = r_plan.returncode == 0
        plan_err = '' if plan_ok else r_plan.stderr[:400]

        return {
            'init_ok'     : True,
            'validate_ok' : True,
            'plan_ok'     : plan_ok,
            'deploy_error': plan_err,
        }

# ── Stratified sample ─────────────────────────────────────────────────────────
if not SKIP_DEPLOY:
    per_level = max(1, DEPLOY_SAMPLE_N // 5)
    df_sample = (
        df_aws
        .groupby('difficulty', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), per_level), random_state=42))
    ).reset_index(drop=True)

    print(f'Deployability check on {len(df_sample)} stratified samples…')
    plan_results = []

    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc='terraform plan'):
        res = terraform_plan(row['content'], row['source_slug'], row['dest_file'])
        plan_results.append(res)

    plan_df = pd.DataFrame(plan_results)
    df_sample = pd.concat([df_sample.reset_index(drop=True), plan_df], axis=1)

    print('\nDeployability summary:')
    print(df_sample[['init_ok','validate_ok','plan_ok']].value_counts().to_string())
    deployable_rate = df_sample['plan_ok'].mean()
    print(f'\nplan pass rate: {deployable_rate:.1%}')
else:
    print('SKIP_DEPLOY=True – terraform plan skipped.')
    df_sample = df_aws.copy()
    for col in ['init_ok','validate_ok','plan_ok','deploy_error']:
        df_sample[col] = None


Deployability check on 50 stratified samples…


/var/folders/mr/k47n2bks0fs1y0cmq922vqbh0000gn/T/ipykernel_44142/662819011.py:83: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), per_level), random_state=42))


terraform plan:   0%|          | 0/50 [00:00<?, ?it/s]


Deployability summary:
init_ok  validate_ok  plan_ok
True     True         False      18

plan pass rate: 0.0%


In [104]:
# ── 10. Aggregate & Export ────────────────────────────────────────────────────
# Builds three output artefacts:
#   (a) full_dataset.csv         – all syntax-clean files across all clouds
#   (b) aws_syntax_clean.csv     – AWS-only, syntax validated
#   (c) aws_deployable.csv       – AWS-only, terraform plan passed (gold set)
#   (d) benchmark_summary.csv    – per-repository statistics

import pandas as pd
from tabulate import tabulate

# (a) Full syntax-clean dataset
df_full_export = df_syntaxok[[
    'source_slug','source_category','primary_cloud','licence_spdx',
    'dest_file','github_url','loc','tokens','difficulty',
    'hcl2_valid','tflint_pass','yaml_valid'
]].copy()
df_full_export.to_csv(DATASET_DIR / 'full_dataset.csv', index=False)

# (b) AWS syntax-clean
df_aws_export = df_aws[[
    'source_slug','source_category','primary_cloud','licence_spdx',
    'dest_file','github_url','loc','tokens','difficulty',
    'hcl2_valid','tflint_pass','yaml_valid'
]].copy()
df_aws_export.to_csv(DATASET_DIR / 'aws_syntax_clean.csv', index=False)

# (c) Deployable gold set (from sampled terraform plan)
if 'plan_ok' in df_sample.columns and df_sample['plan_ok'].notna().any():
    df_deployable = df_sample[df_sample['plan_ok'] == True][[
        'source_slug','source_category','primary_cloud','licence_spdx',
        'dest_file','github_url','loc','tokens','difficulty','content',
        'init_ok','validate_ok','plan_ok'
    ]].copy()
    df_deployable.to_csv(DATASET_DIR / 'aws_deployable.csv', index=False)
    gold_n = len(df_deployable)
else:
    gold_n = 0
    print('No deploy results – aws_deployable.csv skipped.')

# (d) Per-repo summary
summary = (
    df_syntaxok
    .groupby('source_slug')
    .agg(
        total_files  = ('dest_file', 'count'),
        mean_loc     = ('loc', 'mean'),
        mean_tokens  = ('tokens', 'mean'),
        licence      = ('licence_spdx', 'first'),
        cloud        = ('primary_cloud', 'first'),
        category     = ('source_category', 'first'),
    )
    .round(1)
    .reset_index()
    .sort_values('total_files', ascending=False)
)
summary.to_csv(DATASET_DIR / 'benchmark_summary.csv', index=False)

print('=' * 70)
print('BENCHMARK DATASET SUMMARY')
print('=' * 70)
print(f'  Total syntax-clean files : {len(df_syntaxok):>7,}')
print(f'  AWS-targeting files      : {len(df_aws):>7,}')
print(f'  Deployable (gold set)    : {gold_n:>7,}')
print(f'  Repositories included    : {len(summary):>7}')
print()
print(tabulate(summary, headers='keys', tablefmt='rounded_outline', showindex=False))
print()
print('Exports written to:', DATASET_DIR.resolve())


BENCHMARK DATASET SUMMARY
  Total syntax-clean files :     473
  AWS-targeting files      :     201
  Deployable (gold set)    :       0
  Repositories included    :      17

╭───────────────────────────────────────────────┬───────────────┬────────────┬───────────────┬────────────┬─────────┬──────────────────╮
│ source_slug                                   │   total_files │   mean_loc │   mean_tokens │ licence    │ cloud   │ category         │
├───────────────────────────────────────────────┼───────────────┼────────────┼───────────────┼────────────┼─────────┼──────────────────┤
│ ned1313/terraform-tuesdays                    │           223 │       84.2 │         541   │ MIT        │ multi   │ educational      │
│ ValeriiVasianovych/terraform-iac              │            68 │      123.6 │         876.8 │ MIT        │ aws     │ educational      │
│ hashicorp/terraform-guides                    │            42 │      102.2 │         660.4 │ MPL-2.0    │ multi   │ official         │
│ a

In [ ]:
# ── 11. OPTIONAL – Checkov Security Scan (FIXED) ──────────────────────────────
# Fixes:
#   1. Checkov 3.x JSON schema change  (results is sometimes a list)
#   2. ANSI / summary text prepended to stdout breaks json.loads
#   3. Empty deployable set → ck_df has no columns → KeyError on access
#
# Reference metric – Filtered Compliance Rate (FCR):
#   DPIaC-Eval baseline: 8.4%  (Zhang et al., 2025)

import subprocess, json, re, tempfile, pathlib
import pandas as pd
from tqdm.notebook import tqdm

CHECKOV_AVAILABLE = subprocess.run(
    ['which', 'checkov'], capture_output=True
).returncode == 0

# ── Robust JSON extractor ─────────────────────────────────────────────────────
_JSON_RE = re.compile(r'(\{.*\}|\[.*\])', re.DOTALL)

def _extract_json(raw: str):
    """
    Strip ANSI codes and any non-JSON prefix/suffix that Checkov 3.x prints.
    Returns the parsed object or None.
    """
    # Remove ANSI escape codes
    clean = re.sub(r'\x1b\[[0-9;]*m', '', raw)
    # Try full string first (fast path)
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        pass
    # Fallback: find the largest JSON block in stdout
    match = _JSON_RE.search(clean)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return None

# ── Normalise Checkov 2.x / 3.x schema ───────────────────────────────────────
def _parse_checkov_output(obj) -> dict:
    """
    Checkov 2.x  → {"results": {"passed_checks": [...], "failed_checks": [...]}}
    Checkov 3.x  → [{"check_id": ..., "passed": true/false, ...}, ...]
                 OR {"results": {"passed_checks": [...], "failed_checks": [...]}}
    """
    if obj is None:
        return {'checkov_passed': None, 'checkov_failed': None, 'checkov_fcr': None}

    # ── Checkov 3.x list format ────────────────────────────────────────────────
    if isinstance(obj, list):
        passed = sum(1 for c in obj if c.get('passed') is True)
        failed = sum(1 for c in obj if c.get('passed') is False)
        fcr    = (failed == 0) if (passed + failed > 0) else None
        return {'checkov_passed': passed, 'checkov_failed': failed, 'checkov_fcr': fcr}

    # ── Checkov 2.x / 3.x dict format ─────────────────────────────────────────
    if isinstance(obj, dict):
        # Some 3.x builds wrap per-file results under a list at top level
        if 'results' in obj:
            results  = obj['results']
        elif 'check_results' in obj:        # alternate key seen in some 3.x builds
            results  = obj['check_results']
        else:
            results  = obj                  # treat the dict itself as results

        # results may itself be a list of dicts
        if isinstance(results, list):
            return _parse_checkov_output(results)

        passed = len(results.get('passed_checks', []))
        failed = len(results.get('failed_checks', []))
        fcr    = (failed == 0) if (passed + failed > 0) else None
        return {'checkov_passed': passed, 'checkov_failed': failed, 'checkov_fcr': fcr}

    return {'checkov_passed': None, 'checkov_failed': None, 'checkov_fcr': None}

# ── Main scan function ────────────────────────────────────────────────────────
def checkov_scan(content: str) -> dict:
    with tempfile.TemporaryDirectory() as tmpd:
        tf = pathlib.Path(tmpd) / 'main.tf'
        tf.write_text(content, encoding='utf-8')
        r = subprocess.run(
            [
                'checkov', '-f', str(tf),
                '--framework', 'terraform',
                '-o', 'json',
                '--quiet',          # suppress progress bar
                '--compact',        # suppress passed-check details (smaller output)
            ],
            capture_output=True, text=True, timeout=90
        )
        # Checkov exits with code 1 when violations are found – that is expected.
        # Only treat non-0/1 codes as hard errors.
        if r.returncode not in (0, 1):
            return {'checkov_passed': None, 'checkov_failed': None,
                    'checkov_fcr': None, 'checkov_error': r.stderr[:300]}

        obj = _extract_json(r.stdout)
        result = _parse_checkov_output(obj)
        result.setdefault('checkov_error', '')
        return result

# ── Guard: need actual plan results ──────────────────────────────────────────
_has_plan = (
    'plan_ok' in df_sample.columns
    and df_sample['plan_ok'].notna().any()
)

if not CHECKOV_AVAILABLE:
    print('⚠️  checkov not found on $PATH – skipping.\n'
          '   Install with:  pip install checkov')

elif not _has_plan:
    print('⚠️  No terraform plan results found.\n'
          '   Set SKIP_DEPLOY = False and re-run Cell 9 first.')

else:
    df_dep_for_checkov = df_sample[df_sample['plan_ok'] == True].copy()

    if df_dep_for_checkov.empty:
        print('⚠️  No deployable templates in the sample.\n'
              '   Increase DEPLOY_SAMPLE_N or check your AWS credentials.')
    else:
        print(f'Running Checkov on {len(df_dep_for_checkov)} deployable templates…')

        checkov_results = [
            checkov_scan(row['content'])
            for _, row in tqdm(
                df_dep_for_checkov.iterrows(),
                total=len(df_dep_for_checkov),
                desc='Checkov'
            )
        ]

        ck_df = pd.DataFrame(checkov_results)

        # Safety: ensure required columns always exist even if all scans errored
        for col in ('checkov_passed', 'checkov_failed', 'checkov_fcr', 'checkov_error'):
            if col not in ck_df.columns:
                ck_df[col] = None

        df_dep_for_checkov = pd.concat(
            [df_dep_for_checkov.reset_index(drop=True), ck_df], axis=1
        )

        # ── Metrics ──────────────────────────────────────────────────────────
        valid_mask = df_dep_for_checkov['checkov_fcr'].notna()
        n_valid    = valid_mask.sum()

        if n_valid == 0:
            print('⚠️  All Checkov scans returned no parseable output.')
            print('    Checkov stdout sample:')
            # Re-run one file in verbose mode for diagnosis
            sample_content = df_dep_for_checkov.iloc[0]['content']
            with tempfile.TemporaryDirectory() as t:
                p = pathlib.Path(t) / 'main.tf'
                p.write_text(sample_content)
                dbg = subprocess.run(
                    ['checkov', '-f', str(p), '--framework', 'terraform', '-o', 'json'],
                    capture_output=True, text=True, timeout=60
                )
                print('  RETURN CODE:', dbg.returncode)
                print('  STDOUT (first 800 chars):\n', dbg.stdout[:800])
                print('  STDERR (first 400 chars):\n', dbg.stderr[:400])
        else:
            fcr     = df_dep_for_checkov.loc[valid_mask, 'checkov_fcr'].mean()
            avg_p   = df_dep_for_checkov.loc[valid_mask, 'checkov_passed'].mean()
            avg_f   = df_dep_for_checkov.loc[valid_mask, 'checkov_failed'].mean()

            print(f'\n{"─"*50}')
            print(f'  Templates scanned          : {n_valid}')
            print(f'  Filtered Compliance Rate   : {fcr:.1%}')
            print(f'  (DPIaC-Eval baseline: 8.4% | target: >20%)')
            print(f'  Avg checks passed / file   : {avg_p:.1f}')
            print(f'  Avg checks failed / file   : {avg_f:.1f}')
            print(f'{"─"*50}')

        df_dep_for_checkov.to_csv(
            DATASET_DIR / 'aws_deployable_with_checkov.csv', index=False
        )
        print(f'\nSaved → {DATASET_DIR / "aws_deployable_with_checkov.csv"}')


In [ ]:
# ── 12. Dataset Visualisation ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('IaC Benchmark Dataset – Overview', fontsize=14, fontweight='bold')

# (a) Files by repository
ax = axes[0, 0]
src = df_aws['source_slug'].str.split('/').str[-1].value_counts()
ax.barh(src.index, src.values, color='steelblue')
ax.set_title('Files per Repository (syntax-clean)')
ax.set_xlabel('Count')

# (b) Difficulty distribution
ax = axes[0, 1]
diff = df_aws['difficulty'].value_counts().sort_index()
ax.bar(diff.index.astype(str), diff.values,
       color=['#2ECC71','#3498DB','#9B59B6','#F39C12','#E74C3C'])
ax.set_title('Difficulty Distribution (DPIaC-Eval levels)')
ax.set_xlabel('Difficulty Level'); ax.set_ylabel('Count')

# (c) LOC histogram
ax = axes[1, 0]
df_aws['loc'].clip(upper=MAX_LOC).hist(bins=40, ax=ax, color='teal', edgecolor='white')
ax.set_title('Lines of Code Distribution')
ax.set_xlabel('LoC'); ax.set_ylabel('Frequency')

# (d) Cloud target pie
ax = axes[1, 1]
cloud = df_aws['primary_cloud'].value_counts()
ax.pie(cloud.values, labels=cloud.index, autopct='%1.0f%%',
       colors=['#FF9900','#4285F4','#00A1F1','#7FBA00','#EA4335'])
ax.set_title('Cloud Target Distribution')

plt.tight_layout()
plt.savefig(DATASET_DIR / 'dataset_overview.png', dpi=150)
plt.show()
print('Saved: dataset_overview.png')
